# Data Viewer

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import (
    ShortTimeFFT,
    correlate,
    correlation_lags,
    find_peaks,
    welch,
    get_window,
    hilbert,
)
from tqdm.notebook import tqdm
from tritonoa.data.reader import read_hdf5
from tritonoa.data.signal import taper
from tritonoa.data.time import TIME_PRECISION

FIGWIDTH = 14

In [ ]:
sensor = "3dvha"
# sensor = "vla1"
# sensor = "vla2"
# time_start = np.datetime64("2023-12-01 21:06:00", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01 21:08:00", TIME_PRECISION)
# time_start = np.datetime64("2023-12-01 21:10:00", TIME_PRECISION)
# time_end = np.datetime64("2023-12-01 21:12:00", TIME_PRECISION)
time_start = np.datetime64("2023-12-01 22:24:00", TIME_PRECISION)
time_end = np.datetime64("2023-12-01 22:26:00", TIME_PRECISION)

## Filtering Results

In [ ]:
ds = read_hdf5(Path("data/acoustic/denoised") / f"{sensor}.h5").trim(time_start, time_end).filter("bandpass", [15.0, 50.0])
y_orig = ds.data[0]
y_filt = ds.data[1]
y_temp = ds.data[2]
t = ds.time_vector
fs = ds.stats.sampling_rate

In [ ]:
fig, axes = plt.subplots(nrows=3, figsize=(FIGWIDTH, 9), sharex=True)
ax = axes[0]
ax.plot(t, y_orig, label="Original Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal")

ax = axes[1]
ax.plot(t, y_temp, "tab:red", label="Temp Signal")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Template Signal")

ax = axes[2]
ax.plot(t, y_filt, "tab:green", label="Filtered Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Filtered Signal")


plt.tight_layout()
plt.show()

## Pulse Compression Results

In [ ]:
ds = read_hdf5(Path("data/acoustic/denoised") / f"{sensor}_pc.h5").trim(time_start, time_end).filter("bandpass", [15.0, 50.0])
y_orig = ds.data[0]
y_filt = ds.data[1]
pc_orig_type1 = ds.data[2]
pc_orig_type2 = ds.data[3]
pc_dn_type1 = ds.data[4]
pc_dn_type2 = ds.data[5]
t = ds.time_vector
fs = ds.stats.sampling_rate

In [ ]:
fig, axes = plt.subplots(nrows=6, figsize=(FIGWIDTH, 15), sharex=True)

ax = axes[0]
ax.plot(t, y_orig, label="Original Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal")

ax = axes[1]
ax.plot(t, y_filt, label="Filtered Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Filtered Signal")

ax = axes[2]
ax.plot(t, pc_orig_type1, "tab:red", label="PC Original Type 1")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Original Type 1")

ax = axes[3]
ax.plot(t, pc_orig_type2, "tab:green", label="PC Original Type 2")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Original Type 2")

ax = axes[4]
ax.plot(t, pc_dn_type1, "tab:orange", label="PC Denoised Type 1")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Denoised Type 1")

ax = axes[5]
ax.plot(t, pc_dn_type2, "tab:purple", label="PC Denoised Type 2")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Denoised Type 2")

plt.tight_layout()
plt.show()

In [ ]:
ds.write_wav("sample.wav")

## Old

In [ ]:
sig1 = ys
sig2 = ys
xcorr = correlate(sig1, sig2, mode="full") / np.sqrt(np.sum(sig1**2) * np.sum(sig2**2))
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.title("Autocorrelation of Strike Signal")
plt.tight_layout()
plt.show()

sig1 = yw1
sig2 = yw1
xcorr = correlate(sig1, sig2, mode="full") / np.sqrt(np.sum(sig1**2) * np.sum(sig2**2))
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.title("Autocorrelation of Type 1 Whale Signal")
plt.tight_layout()
plt.show()

sig1 = yw2
sig2 = yw2
xcorr = correlate(sig1, sig2, mode="full") / np.sqrt(np.sum(sig1**2) * np.sum(sig2**2))
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.title("Autocorrelation of Type 2 Whale Signal")
plt.tight_layout()
plt.show()

sig1 = ys
sig2 = yw1
xcorr = correlate(sig1, sig2, mode="full") / np.sqrt(np.sum(sig1**2) * np.sum(sig2**2))
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation of Strike and Type 1 Whale Signals")
plt.tight_layout()
plt.show()

sig1 = ys
sig2 = yw2
xcorr = correlate(sig1, sig2, mode="full") / np.sqrt(np.sum(sig1**2) * np.sum(sig2**2))
lags = np.arange(-len(sig1) + 1, len(sig2))
plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(lags, xcorr)
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation of Strike and Type 2 Whale Signals")
plt.tight_layout()
plt.show()

In [ ]:
xcorr = correlate(y, yw1, mode="same") / np.sqrt(np.sum(yw1**2) * np.sum(y**2))
lags = correlation_lags(len(y), len(yw1), mode="same")

peaks = find_peaks(xcorr, height=0.5, distance=int(1.0 * fs))[0]
peak_amps = xcorr[peaks]

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, y / np.max(np.abs(y)))
plt.plot(t, xcorr / np.max(np.abs(xcorr)), label="Cross-correlation between signal & whale")
[plt.axvline(t[pk], color="r") for pk in peaks]
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Full Signal and Whale Type 1")
plt.tight_layout()
plt.show()

xcorr = correlate(y, yw2, mode="same") / np.sqrt(np.sum(yw2**2) * np.sum(y**2))
lags = correlation_lags(len(y), len(yw2), mode="same")

peaks = find_peaks(xcorr, height=0.5, distance=int(1.0 * fs))[0]
peak_amps = xcorr[peaks]

plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, y / np.max(np.abs(y)))
plt.plot(t, xcorr / np.max(np.abs(xcorr)), label="Cross-correlation between signal & whale")
[plt.axvline(t[pk], color="r") for pk in peaks]
plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Full Signal and Whale Type 2")
plt.tight_layout()
plt.show()


xcorr = correlate(y, ys, mode="same") / np.sqrt(np.sum(ys**2) * np.sum(y**2))
lags = correlation_lags(len(y), len(ys), mode="same")
cf = xcorr / np.max(np.abs(xcorr))

peaks = find_peaks(cf, height=0.3, distance=int(1.1 * fs))[0]
peak_amps = cf[peaks]


plt.figure(figsize=(FIGWIDTH, 3))
plt.plot(t, y / np.max(np.abs(y)))
plt.plot(t, cf, label="Cross-correlation between signal & strike")
plt.plot(t[peaks], peak_amps, "ro", label="Peaks")
# [plt.axvline(t[pk], color="r") for pk in peaks]

plt.grid()
plt.xlabel("Lag")
plt.ylabel("Cross-correlation")
plt.title("Cross-correlation between Full Signal and Strike")
plt.tight_layout()
plt.show()

In [ ]:
# whale_call_inds = [24]
# whale_call_inds = [3]
whale_call_inds = [3, 8, 9, 14, 19, 20, 22]
# whale_call_inds = [3, 4, 8, 9, 14, 20, 25, 30, 35, 40, 41, 46, 51, 52, 54]

plt.figure(figsize=(FIGWIDTH, 4))
for pk, ind in enumerate(peaks):
    start = int(ind - 1.0 * fs)
    end = int(ind + 1.0 * fs)
    segment = y[start:end]
    f, psd = welch(segment, fs=fs, nperseg=min(8192, len(segment)), nfft=2 ** 13)
    if pk in whale_call_inds:
        plt.plot(f, psd, "r--")
    else:
        plt.plot(f, psd, "k-")
plt.xlim([10, 50])
# plt.legend()
plt.show()

In [ ]:
# buffer_start = 0.9
# buffer_end = 0.7
buffer_start = 1.1
buffer_end = 0.9

fig = plt.figure(figsize=(FIGWIDTH, 4))
for pk, ind in enumerate(peaks):
    start = int(ind - buffer_start * fs)
    end = int(ind + buffer_end * fs)

    if start < 0 or end > len(y):
        continue
    
    segment = y[start:end]
    
    # Skip if segment is too short or empty
    if len(segment) == 0:
        continue
    
    # Create time array based on actual segment
    actual_start_buffer = (ind - start) / fs
    tseg = np.arange(-actual_start_buffer, (end - ind) / fs, 1 / fs)[:len(segment)]
    
    if pk in whale_call_inds:
        style = "r-"
    else:
        style = "k-"
    plt.plot(tseg, segment / np.max(np.abs(segment)) + pk, style)
    plt.gca().invert_yaxis()
plt.show()

fig = plt.figure(figsize=(FIGWIDTH, 4))
for pk, ind in enumerate(peaks):
    start = int(ind - buffer_start * fs)
    end = int(ind + buffer_end * fs)
    segment = y[start:end]
    if start < 0 or end > len(y):
        continue

    segment = y[start:end]
    
    # Skip if segment is too short or empty
    if len(segment) == 0:
        continue
    
    # Create time array based on actual segment
    actual_start_buffer = (ind - start) / fs
    tseg = np.arange(-actual_start_buffer, (end - ind) / fs, 1 / fs)[:len(segment)]
    
    if pk in whale_call_inds:
        style = "r-"
        continue
    else:
        style = "k-"
    plt.plot(tseg, segment / np.max(np.abs(segment)), style)
plt.show()

In [ ]:
print(len(tseg))

segments = []
for pk, ind in enumerate(peaks):
    start = int(ind - buffer_start * fs)
    end = int(ind + buffer_end * fs)
    if start < 0 or end > len(y):
        continue
    
    segment = y[start:end]
    
    segments.append(segment)

template = np.median(np.array(segments), axis=0)

tap = taper(len(template), max_percentage=0.05)
# # window = "tukey"
# # taper_len = len(template) // 2
# # taper = get_window(window, taper_len)
# # print(len(taper), len(tseg))
# # taper = np.concatenate((taper[:taper_len], np.zeros(len(tseg) - len(taper)), taper[:taper_len]))
template *= tap

plt.figure(figsize=(FIGWIDTH, 4))
[plt.plot(tseg, segment, "k-") for segment in segments]
plt.plot(tseg, template, "r-")
# plt.plot(tseg, tap, "b--")
plt.show()

# Subtraction

In [ ]:
y_filtered = y.copy()
x = np.zeros_like(y)

for pk, ind in tqdm(enumerate(peaks), total=len(peaks)):
    start = int(ind - buffer_start * fs)
    end = int(ind + buffer_end * fs)
    y_segment = y[start:end].copy()
    if y_segment.shape[0] != len(template):
        continue

    e = y_segment - template
    # e = spectral_subtraction(y_segment, template)
    y_filtered[start:end] = e
    x[start:end] = template


xcorr1_orig = correlate(y, yw1, mode="same") / np.sqrt(np.sum(y**2) * np.sum(yw1**2))
xcorr1_filt = correlate(y_filtered, yw1, mode="same") / np.sqrt(np.sum(y_filtered**2) * np.sum(yw1**2))
xcorr2_orig = correlate(y, yw2, mode="same") / np.sqrt(np.sum(y**2) * np.sum(yw2**2))
xcorr2_filt = correlate(y_filtered, yw2, mode="same") / np.sqrt(np.sum(y_filtered**2) * np.sum(yw2**2))

In [ ]:
fig, axs = plt.subplots(nrows=4, figsize=(FIGWIDTH, 8), sharex=True)
ax = axs[0]
ax.plot(t, y, "tab:red", label="Original Signal")
ax.plot(t, y_filtered, "tab:green", label="Error (Filtered Signal)")
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal and Filtered Signal")
ax.legend()

ax = axs[1]
ax.plot(t, y, "tab:red", label="Original Signal")
ax.plot(t, x, "tab:green", label="Filtered Signal")
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal and Filtered Signal (Template)")
ax.legend()

ax = axs[2]
ax.plot(t, np.abs(hilbert(xcorr1_filt)), "tab:green", label="Filtered Signal")
ax.plot(t, np.abs(hilbert(xcorr1_orig)), "tab:red", label="Original Signal")
ax.set_title("Cross-correlation between Filtered Signal and Whale Type 1")
ax.legend()

ax = axs[3]
ax.plot(t, np.abs(hilbert(xcorr2_filt)), "tab:green", label="Filtered Signal")
ax.plot(t, np.abs(hilbert(xcorr2_orig)), "tab:red", label="Original Signal")
ax.set_title("Cross-correlation between Filtered Signal and Whale Type 2")
ax.legend()

for ax in axs:
    ax.grid()

plt.show()

In [ ]:
window = "hann"
nperseg = 2048
hop = 512
flim = [15, 30]


STFT = ShortTimeFFT(fs=fs, hop=hop, mfft=nperseg, win=get_window(window, nperseg))
Zxx_orig = STFT.stft(y)
Zxx_orig_db = 20 * np.log10(np.abs(Zxx_orig))
tvec = STFT.t(len(y))
f = STFT.f

vmax = np.max(Zxx_orig_db)
vmin = vmax - 60
print(vmin, vmax)

plt.figure(figsize=(FIGWIDTH, 12))
plt.subplot(311)
plt.pcolormesh(tvec, f, Zxx_orig_db, shading="gouraud", vmin=vmin, vmax=vmax)
plt.ylim(flim)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("STFT of Original Signal")


STFT = ShortTimeFFT(fs=fs, hop=hop, mfft=nperseg, win=get_window(window, nperseg))
Zxx_filt = STFT.stft(y_filtered)
Zxx_filt_db = 20 * np.log10(np.abs(Zxx_filt))
tvec = STFT.t(len(y_filtered))
f = STFT.f

plt.subplot(312)
plt.pcolormesh(tvec, f, Zxx_filt_db, shading="gouraud", vmin=vmin, vmax=vmax)
plt.ylim(flim)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("STFT of Filtered Signal")

Zxx_diff = Zxx_filt_db - Zxx_orig_db

plt.subplot(313)
plt.pcolormesh(tvec, f, Zxx_diff, shading="gouraud", cmap="bwr", vmin=-20, vmax=20)
plt.ylim(flim)
plt.colorbar(label="Magnitude Difference (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Difference")
plt.tight_layout()
plt.show()

